# Juzgando por la portada 

Este notebook pretende mostrar la implementación y funcionamiento del proyecto *Juzgando por la portada* de la asignatura de Procesamiento de Imágenes Digitales (PID) de la Universidad de Sevilla. 

## Pasos previos
Si pretendes ejecutar este notebook, se recomienda encarecidamente usar CUDA para poder habilitar el entrenamiento con la GPU.
CUDA tiene que estar instalado de antemano.

TODO: actualizar 

> NOTA: se recomienda encarecidamente usar Linux directamente debido a las conocidas complicaciones de usar CUDA en Windows. Si aun así el usuario quisiera seguir usando Windows, se anima al usuario a encontrar soluciones y/o vías alternativas en foros o guías por su propia cuenta. 

# Imports

En esta celda puedes encontrar TODOS los imports que vas a necesitar a lo largo del notebook, asegúrate de que esta celda corre correctamente para evitar problemas futuros en la ejecución del notebook.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import clear_output
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random
from tensorflow.keras.preprocessing.image import load_img, img_to_array

import lib

I0000 00:00:1776251400.871011    9752 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776251401.240811    9752 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776251404.856065    9752 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.



Verificando qué imágenes existen...
Total imágenes en CSV: 415966
Imágenes que existen: 0
ADVERTENCIA: No se encontraron imágenes.
Asegúrate de haber ejecutado primero: python scripts/descarga_imagenes.py


ValueError: No hay imágenes disponibles para entrenar. Ejecuta el script de descarga primero.

# Definición del modelo

Podemos observar que tenemos un modelo con 3 bloques, dos capas convolucionales con función de activación ReLU y MaxPooling, un tercer bloque con GAP y finalmente una última capa con función de activación sigmoide para tener una salida binaria multietiqueta. 

In [ ]:
def intersperse(list, delimiter):
    try:
        it = iter(iterable)
        yield next(it)
        for x in it:
            yield delimiter
            yield x
    except StopIteration:
        return

def createModel(convLayers=3, firstConvFilterCount=32, denseLayers=1, firstDenseNeuronCount=256):
    model = models.Sequential(
        [
            layers.Input(shape=(*IMG_SIZE, 3)),
        ]
        +
        # Capas convolucionales
        list(intersperse(
            # Cada capa convolucional tiene el doble de filtros de la anterior
            [layers.Conv2D(firstConvFilterCount * (2 ** i), (3, 3), padding='same', activation='relu')
            for i in range(0,convLayers)],
            # Between each is a MaxPooling2D layer
            layers.MaxPooling2D(2, 2)
        ))
        +
        # GAP final para reducir dimensionalidad
        [
            layers.GlobalAveragePooling2D(),
        ]
        +
        [ 
            # Cada capa densa tiene la mitad de filtros de la anterior
            layers.Dense(firstDenseNeuronCount * (2 ** -i), (3, 3), padding='same', activation='relu')
            for i in range(0,denseLayers)
        ]
        +
        # En mixed precision dejamos la salida en float32 para mayor estabilidad numérica (esto me lo ha dicho la IA¿?)
        [
            layers.Dense(len(genre_columns), activation='sigmoid', dtype='float32')
        ]
    )

    # Compilar el modelo
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()

    return model

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 96, 144, 24)    │           672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 48, 72, 24)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 48, 72, 48)     │        10,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 36, 48)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 24, 36, 96)     │        41,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 96)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 19)             │         2,451 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,523 (263.76 KB)

 Trainable params: 67,523 (263.76 KB)

 Non-trainable params: 0 (0.00 B)

# Busqueda en cuadrícula

In [ ]:
def generate_text(hyperparams, metrics):
    text = ""

    text += "=== HIPERPARÁMETROS ===\n"

    for k, v in hyperparams.items():
        text += f"  {k}: {v}\n"

    text += "\n=== MÉTRICAS ===\n"
    for k, v in metrics.items():
        text += f"  {k}: {v:.4f}\n"

    save_result(path, )

    keras_path = path.replace(".txt", ".keras")
    model.save(keras_path)


possibilities = [
    # Convolutional layers
    [2,3,4],
    # Filter count of first convolutional layer
    [32,64]
    # Dense layers
    [1,2]
    # Neuron count of first dense neuron
    [256,512]
]

permutations = list(itertools.product(*possibilities))

for [convLayers, firstConvFilterCount, denseLayers, firstDenseNeuronCount] in permutations:
    print("Entrenando modelo con: ")
    print(f" - Capas convolucionales: {convLayers}")
    print(f" - Cantidad de filtros en la primera capa convolucional: {firstConvFilterCount}")
    print(f" - Capas densas: {denseLayers}")
    print(f" - Cantidad de neuronas en la primera capa densa: {firstDenseNeuronCount}")
    
    path = result_path(convLayers, firstConvFilterCount, denseLayers, firstDenseNeuronCount)

    if os.path.exists(path + ".txt"):
        print(f"Saltando {os.path.basename(path)} (ya entrenado)")
        continue

    model = createModel(convLayers, firstConvFilterCount, denseLayers, firstDenseNeuronCount)
    history = train_model(model)
    represent_data(history)
    [exact_match, precision_micro, recall_micro, f1_micro, precision_macro, recall_macro, f1_macro] = evaluate_model(model)
    predict_single_image(model)

    save_result(path,
        generate_text({
            "convLayers": convLayers,
            "firstConvFilterCount": firstConvFilterCount,
            "denseLayers": denseLayers,
            "firstDenseNeuronCount": firstDenseNeuronCount,
        },
        {
            "exact_match": exact_match,
            "precision_micro": precision_micro,
            "recall_micro": recall_micro,
            "f1_micro": f1_micro,
            "precision_macro": precision_macro,
            "recall_macro": recall_macro,
            "f1_macro": f1_macro,
        })
        model,
    )
    

# Ejemplo de funcionamiento

En la siguiente celda podemos observar cómo funciona el modelo, junto a las etiquetas predichas y la etiqueta real. Se muestra también las mejores 5 predicciones. 